# DP-ProtOn: interactive demo

**Dirichlet-Process-guided Prototype Online learner** — a minimal, gradient-free, single-pass continual-learning classifier with zero-retraining knowledge editing.

This notebook demonstrates, in order:
1. **Streaming learning** on digits (PCA-whitened 30-D, frozen features)
2. **Continual learning** on Split-digits (5 tasks x 2 classes, no revisit)
3. **Knowledge editing**: delete / relabel / inject a class — no retraining

Model: 3 components only (CRP basin creation, online denoised centroids, Parzen density readout). ~250 lines of NumPy. No PyTorch needed.

## 0. Setup

In Colab: `Runtime -> Change runtime type -> CPU` is fine.

In [ ]:
!pip install -q numpy scikit-learn matplotlib
# grab the package straight from the repo (or: pip install dp-proton)
!git clone -q https://github.com/YOUR_USERNAME/dp-proton.git /content/dp-proton 2>/dev/null || echo 'clone failed - use the repo path below'
import sys, os
REPO = '/content/dp-proton' if os.path.isdir('/content/dp-proton') else '..'
sys.path.insert(0, REPO)
from dp_proton import DPProtOn
print('DP-ProtOn imported OK')

## 1. Streaming learning in 5 lines

Learn the stream once (no epochs, no gradient), then classify.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

d = load_digits()
X = PCA(n_components=30, whiten=True, random_state=0).fit_transform(
    StandardScaler().fit_transform(d.data))
y = d.target
rng = np.random.default_rng(0)
idx = rng.permutation(len(X)); X, y = X[idx], y[idx]
n = int(0.7 * len(X))

model = DPProtOn(alpha=1.0, n_sub=1)
model.fit_stream(X[:n], y[:n])          # single pass, no revisit
acc = (model.classify(X[n:]) == y[n:]).mean()
print(f'5-line DP-ProtOn: test ACC = {acc:.4f}')
print(f'basins: {model.summary()["n_basins"]}, prototypes: {model.summary()["n_prototypes"]}')

## 2. Continual learning: Split-digits, task by task

Each task brings 2 new classes. Old data is never revisited. The model opens new basins (CRP rule) without touching old ones — near-zero forgetting by design.

In [ ]:
def split_digits(seed=0):
    d = load_digits(); X, y = d.data, d.target
    r = np.random.default_rng(seed); idx = r.permutation(len(X))
    X, y = X[idx], y[idx]
    n_pre = int(0.3 * len(X))
    sc = StandardScaler().fit(X[:n_pre])
    pca = PCA(n_components=30, whiten=True, random_state=seed).fit(sc.transform(X[:n_pre]))
    Xr = pca.transform(sc.transform(X[n_pre:])); yr = y[n_pre:]
    tasks = []
    for k in range(5):
        cls = (2*k, 2*k+1); m = np.isin(yr, cls)
        Xk, yk = Xr[m], yr[m]
        p = np.random.default_rng(100+k).permutation(len(Xk))
        Xk, yk = Xk[p], yk[p]
        s = int(0.7*len(Xk)); tasks.append((Xk[:s], yk[:s], Xk[s:], yk[s:]))
    return tasks

tasks = split_digits()
model = DPProtOn(alpha=1.0, n_sub=1)
acc_history = []
for t, (Xtr, ytr, Xte, yte) in enumerate(tasks):
    model.fit_stream(Xtr, ytr)
    # evaluate on ALL classes seen so far
    Xall = np.vstack([tasks[k][2] for k in range(t+1)])
    yall = np.concatenate([tasks[k][3] for k in range(t+1)])
    a = (model.classify(Xall) == yall).mean()
    acc_history.append(a)
    print(f'task {t+1} (classes {2*t}-{2*t+1}): ACC on all seen = {a:.4f}, basins = {len(model.basins)}')

import matplotlib.pyplot as plt
plt.plot(range(1, 6), acc_history, 'o-')
plt.xlabel('task'); plt.ylabel('ACC (all seen classes)'); plt.title('Single-pass continual learning, no forgetting curve')
plt.ylim(0.5, 1.0); plt.grid(alpha=0.3); plt.show()

## 3. Knowledge editing: three surgeries, zero retraining

Knowledge lives in explicit, locatable basins (not distributed weights) — so we can edit it directly. A softmax MLP cannot do any of this without retraining the whole network.

In [ ]:
X = np.vstack([t[2] for t in tasks]); y = np.concatenate([t[3] for t in tasks])
Xtr = np.vstack([t[0] for t in tasks]); ytr = np.concatenate([t[1] for t in tasks])

# --- 3a. DELETE class 9 ---
model = DPProtOn(alpha=1.0, n_sub=1)
for Xtr_, ytr_, _, _ in tasks: model.fit_stream(Xtr_, ytr_)
nov_before = model.novelty(X[y==9]).mean()
model.delete_class(9)
preds = model.classify(X)
acc9 = (preds[y==9] == 9).mean()
others = y != 9
acc_other = (preds[others] == y[others]).mean()
nov_after = model.novelty(X[y==9]).mean()
print(f'DELETE class 9: class-9 ACC = {acc9:.3f} (gone), others ACC = {acc_other:.3f} (untouched)')
print(f'              novelty score {nov_before:.2f} -> {nov_after:.2f}  (model now flags 9s as unknown)')

# --- 3b. RELABEL 7 -> 99 ---
model = DPProtOn(alpha=1.0, n_sub=1)
for Xtr_, ytr_, _, _ in tasks: model.fit_stream(Xtr_, ytr_)
p0 = model.classify(X)
model.relabel_class(7, 99)
p1 = model.classify(X)
identical = float((np.where(p1==99, 7, p1) == p0).mean())
print(f'RELABEL 7->99: remap-identical = {identical:.4f} (decision geometry byte-identical)')

# --- 3c. INJECT class 9 from 20 examples ---
model = DPProtOn(alpha=1.0, n_sub=1)
for Xtr_, ytr_, _, _ in tasks: model.fit_stream(Xtr_, ytr_)
model.delete_class(9)
p0 = model.classify(X)
Xk, yk = Xtr[ytr==9][:20], np.full(20, 9)
model.inject_class(Xk, yk)
p1 = model.classify(X)
acc9 = (p1[y==9] == 9).mean()
unchanged = float((p1[y!=9] == p0[y!=9]).mean())
print(f'INJECT class 9 from 20 samples: class-9 ACC = {acc9:.3f} (recovered), others unchanged = {unchanged:.4f}')

## 4. Why it works (30 seconds)

- **CRP basin creation**: opening a new basin is a Bayesian decision — `log n_i + log N(x | c_i, Σ_i)` vs the rejection baseline `log α + log p₀(x)`. No threshold to tune.
- **Online denoised centroids**: running means under competitive assignment; each basin keeps a few (here: 1) sub-centroids that denoise the raw stream.
- **Parzen readout**: `s_i(x) = logsumexp_k(-||x - m_k||² / 2h²)`; the label is the argmax over per-label aggregated densities.

Full story (including what was tried and *falsified*): `paper/TECH_REPORT.md`. All paper numbers reproducible via `experiments/run_all.py`.